# Notebook 3 — Summariser Model Evaluation

This notebook compares three open-source summarisation models so you can pick the best one for your use case:

| Model | Size | Speed | Quality |
|---|---|---|---|
| `facebook/bart-large-cnn` | ~1.6 GB | Medium | Best for news |
| `sshleifer/distilbart-cnn-12-6` | ~300 MB | Fast | Good quality, lighter |
| `google/pegasus-cnn_dailymail` | ~2.3 GB | Slow | Strong abstractive summaries |

**Sources:**  
- BART: https://huggingface.co/facebook/bart-large-cnn  
- DistilBART: https://huggingface.co/sshleifer/distilbart-cnn-12-6  
- PEGASUS: https://huggingface.co/google/pegasus-cnn_dailymail  

**Input:** `smart_news_summarised.csv` (from Notebook 2)  
**Output:** `model_comparison.csv` with all three models' summaries side by side

In [ ]:
# %pip install transformers torch pandas rouge-score sentencepiece

In [ ]:
import pandas as pd
import torch
from transformers import pipeline
import time

device = 0 if torch.cuda.is_available() else -1
print(f"Device: {'GPU' if device == 0 else 'CPU'}")

In [ ]:
# ─────────────────────────────────────────────
# CONFIG: Select how many articles to compare
# Start with 5 — comparing all 3 models on 30+ articles is slow on CPU.
# ─────────────────────────────────────────────
N_ARTICLES       = 5
INPUT_CSV        = 'smart_news_summarised.csv'
INPUT_CHAR_LIMIT = 3000

MODELS = {
    'bart_large':   'facebook/bart-large-cnn',
    'distilbart':   'sshleifer/distilbart-cnn-12-6',
    'pegasus':      'google/pegasus-cnn_dailymail',
}

In [ ]:
df = pd.read_csv(INPUT_CSV).head(N_ARTICLES)
print(f"Testing on {len(df)} articles")

In [ ]:
def run_model(model_name, texts, max_tokens=130, min_tokens=30):
    """Load a model, summarise all texts, return summaries + timing."""
    print(f"\nLoading: {model_name}")
    print("(Will download if not cached)")
    
    pipe = pipeline("summarization", model=model_name, device=device)
    
    summaries = []
    start = time.time()
    
    for i, text in enumerate(texts):
        if not isinstance(text, str) or len(text.strip()) < 50:
            summaries.append("Not enough text.")
            continue
        text_input = text[:INPUT_CHAR_LIMIT]
        try:
            out = pipe(text_input, max_length=max_tokens, min_length=min_tokens, do_sample=False)
            raw = out[0]['summary_text'].strip()
            words = raw.split()
            summaries.append(' '.join(words[:60]) + ('...' if len(words) > 60 else ''))
        except Exception as e:
            summaries.append(f"Error: {str(e)[:50]}")
        print(f"  [{i+1}/{len(texts)}] done")
    
    elapsed = time.time() - start
    print(f"  Total time: {elapsed:.1f}s | Avg per article: {elapsed/len(texts):.1f}s")
    
    del pipe  # Free memory before loading next model
    return summaries, elapsed

In [ ]:
# ─────────────────────────────────────────────
# RUN ALL 3 MODELS
# Uses full_news if available, falls back to rss_summary
# ─────────────────────────────────────────────
texts = []
for _, row in df.iterrows():
    t = row.get('full_news', '')
    if not isinstance(t, str) or len(t.strip()) < 50:
        t = row.get('rss_summary', '')
    texts.append(t)

results = {}
timings = {}

for model_key, model_name in MODELS.items():
    summaries, elapsed = run_model(model_name, texts)
    results[model_key] = summaries
    timings[model_key] = elapsed

In [ ]:
# ─────────────────────────────────────────────
# BUILD COMPARISON DATAFRAME
# ─────────────────────────────────────────────
compare_df = df[['website_name', 'title']].copy()

for model_key in MODELS:
    compare_df[f'summary_{model_key}'] = results[model_key]

compare_df.to_csv('model_comparison.csv', index=False, encoding='utf-8')
print("Saved: model_comparison.csv")

print("\n--- Speed Summary ---")
for model_key, elapsed in timings.items():
    print(f"  {model_key:20s}: {elapsed:.1f}s total | {elapsed/N_ARTICLES:.1f}s per article")

In [ ]:
# ─────────────────────────────────────────────
# SIDE-BY-SIDE PREVIEW
# ─────────────────────────────────────────────
for i, row in compare_df.iterrows():
    print(f"\n{'='*70}")
    print(f"TITLE: {row['title']}")
    print(f"SOURCE: {row['website_name']}")
    for model_key in MODELS:
        print(f"\n  [{model_key.upper()}]")
        print(f"  {row[f'summary_{model_key}']}")

print(f"\n{'='*70}")
print("\nRecommendation:")
print("  On CPU with limited RAM: use distilbart (fastest, smallest)")
print("  On GPU or for best quality: use bart-large-cnn")
print("  For more abstractive/rephrased summaries: use pegasus")